# Replanteamiento y Explicaciones del Archivo `docker-compose.yml`

Este archivo `docker-compose.yml` define la configuración de múltiples servicios para un proyecto con varias tecnologías, incluyendo SQLite, Node.js, PHP, TypeScript, Avalanche Blockchain y OWASP ZAP para ciberseguridad. Cada uno de los servicios tiene su justificación técnica, así como la configuración de volúmenes persistentes y redes.

## Redes
### Definición de la red `transcendence`

- **driver: bridge**: Se ha utilizado el driver `bridge` porque es el comportamiento predeterminado de Docker, lo que permite que los contenedores en esta red se comuniquen entre sí de manera eficiente mientras mantienen aislamiento de otros contenedores que no están en esta red. Esta red facilita la conexión de todos los servicios.

```yaml
networks:
  transcendence:
    name: transcendence
    driver: bridge
```

## Servicios

### 1. `sqlite`: Base de datos

SQLite es una base de datos ligera y fácil de configurar, ideal para entornos de desarrollo y pequeños proyectos.

- **build**: Se especifica un contexto de construcción desde `dockers/sqlite/.`, lo que permite crear la imagen desde un Dockerfile personalizado.
- **image**: Se usa una imagen de SQLite personalizada `nouchka/sqlite3`, pero este paso es opcional si se cuenta con un Dockerfile adecuado.
- **volumes**: Se ha montado un volumen persistente `sqlite_data` en el directorio `/var/lib/sqlite` para garantizar que los datos de la base de datos persistan incluso si el contenedor se detiene o elimina.
- **command**: El comando `tail -f /dev/null` mantiene el contenedor en ejecución para acceder a la base de datos en cualquier momento.

```yaml
  sqlite:
    build: dockers/sqlite/.
    container_name: sqlite
    image: nouchka/sqlite3
    volumes:
      - sqlite_data:/var/lib/sqlite
    command: tail -f /dev/null
    networks:
      - transcendence
    restart: always
```

### 2. `backend`: API de Node.js

Este servicio representa la API que se comunica con la base de datos y otros servicios.

- **build**: Se construye desde `dockers/backend/.`, donde está definido el Dockerfile del backend.
- **volumes**: Se monta el volumen persistente `backend_data` en `/usr/src/app/data` para almacenar datos esenciales del backend.
- **command**: Se ejecuta `npm start` para iniciar la aplicación Node.js.
- **depends_on**: Define que este servicio depende de que `sqlite` y `avalanche` estén en funcionamiento antes de iniciar.
- **restart**: Se ha establecido `unless-stopped` para que se reinicie automáticamente si falla, pero se detendrá manualmente si el administrador lo decide.

```yaml
  backend:
    build: dockers/backend/.
    container_name: app
    working_dir: /usr/src/app
    volumes:
      - backend_data:/usr/src/app/data
    command: npm start
    ports:
      - "3000:3000"
    depends_on:
      - sqlite
      - avalanche
    networks:
      - transcendence
    restart: unless-stopped
```

### 3. `php`: Servidor PHP

PHP sigue siendo útil para algunos componentes legados o específicos del proyecto.

- **build**: Se especifica la construcción desde `dockers/php/.` para permitir personalizar la configuración del servidor PHP.
- **image**: Se utiliza la imagen `php:8.1-apache`, la cual ya incluye Apache como servidor web.
- **volumes**: Aquí se ha montado la carpeta `./src/dockers/php` desde el host a `/var/www/html`, donde se alojan los archivos PHP, aunque no se ha definido un volumen persistente en este caso porque no se requiere la persistencia en el host en esta fase del proyecto.
- **volumesII**: Creamos volumen persistente por error al copiar la configuración del servidor.
- **ports**: El puerto 8080 del host está mapeado al puerto 80 del contenedor para servir aplicaciones web.

```yaml
  php:
    build: dockers/php/.
    container_name: php
    image: php:8.1-apache
    volumes:
      - ./src/dockers/php:/var/www/html
    ports:
      - "8080:80"
    networks:
      - transcendence
    restart: always
```

### 4. `frontend`: Frontend en TypeScript con Tailwind

Este servicio se encarga de servir la interfaz gráfica de usuario del proyecto.

- **build**: Se utiliza `dockers/frontend/.` como contexto de construcción para el frontend.
- **image**: La imagen `node:18-alpine` es liviana y está optimizada para entornos de producción de Node.js.
- **volumes**: Se ha configurado el volumen persistente `frontend_data` en `/usr/src/app`.
- **command**: Se ejecuta el comando `npm run dev` para iniciar el entorno de desarrollo del frontend.

```yaml
  frontend:
    build: dockers/frontend/.
    container_name: frontend
    image: node:18-alpine
    volumes:
      - frontend_data:/usr/src/app
    command: npm run dev
    ports:
      - "3001:3001"
    networks:
      - transcendence
    restart: unless-stopped
    depends_on:
      - backend
```

### 5. `avalanche`: Blockchain (Avalanche)

Avalanche es una plataforma de blockchain diseñada para aplicaciones descentralizadas. Este servicio gestiona la cadena de bloques y su red.

- **image**: Se utiliza la imagen oficial `avaplatform/avalanchego:latest`.
- **volumes**: Se monta el volumen persistente `blockchain_data` en `/root/.avalanchego` para guardar la configuración y datos de la blockchain.
- **ports**: Se exponen dos puertos, `9650` para el endpoint JSON-RPC y `9651` para la comunicación entre nodos (P2P).

```yaml
  avalanche:
    build: dockers/blockchain/.
    container_name: blockchain
    image: avaplatform/avalanchego:latest
    volumes:
      - blockchain_data:/root/.avalanchego
    ports:
      - "9650:9650"
      - "9651:9651"
    command: ["/avalanchego/avalanchego"]
    networks:
      - transcendence
    restart: unless-stopped
```

### 6. `security`: OWASP ZAP para ciberseguridad

OWASP ZAP se utiliza para realizar pruebas de seguridad automatizadas en aplicaciones web.

- **image**: `softwaresecurityproject/zap-stable` es una imagen estable de OWASP ZAP.
- **volumes**: Se monta la carpeta `./tools/zap_rules` del host en `/zap/wrk/` para cargar reglas personalizadas de escaneo.

```yaml
  security:
    build: dockers/security/.
    container_name: security
    image: softwaresecurityproject/zap-stable
    volumes:
      - ./tools/zap_rules:/zap/wrk/
    ports:
      - "8081:8081"
    networks:
      - transcendence
    restart: unless-stopped
```

## Volúmenes Persistentes

En este proyecto, hemos definido volúmenes para persistir datos críticos de algunos servicios. Estos volúmenes aseguran que los datos no se pierdan cuando los contenedores se reinician o eliminan.

```yaml
volumes:
  sqlite_data:
    name: sqlite_data
    driver: local
    driver_opts:
      type: none
      device: "/Users/usuario/data/sqlite"
      o: bind

  backend_data:
    name: backend_data
    driver: local
    driver_opts:
      type: none
      device: "/Users/usuario/data/app"
      o: bind

  frontend_data:
    name: frontend_data
    driver: local
    driver_opts:
      type: none
      device: "/Users/usuario/data/frontend"
      o: bind

  blockchain_data:
    name: blockchain_data
    driver: local
    driver_opts:
      type: none
      device: "/Users/usuario/data/blockchain"
      o: bind
```

## Justificación Técnica
### Por qué hemos elegido estos servicios y configuración
- **SQLite**: Es una base de datos ligera, rápida y fácil de usar, ideal para proyectos pequeños o medianos, y que no requiere configuración de servidor como MySQL o PostgreSQL.
- **Node.js (Backend)**: Para crear una API eficiente y no bloqueante



***
***
***

# Configuraciones Parte II



### 1. `frontend`: Frontend en TypeScript con Tailwind

Deesplegamos frontend en producción sin un servidor adicional (como Nginx), podremos generar los archivos estáticos con Vite y configurarlo para que se sirvan desde Apache (el mismo contenedor PHP que tenemos):

Generar archivos estáticos del frontend:

- 1. **Dockerfile**: Ejecutar el comando `npm run build` en el contenedor del frontend para generar los archivos optimizados. Esto colocará los archivos en la carpeta `dist`.

```yaml
    FROM node:18-alpine

    WORKDIR /usr/src/app

    COPY package*.json ./

    RUN npm install

    COPY . .

    EXPOSE 3001

    CMD ["npm", "run", "dev"]
```

- 2. **Docker  Compose**: Mover los archivos generados al servidor web (Apache):

    Como no usamos `Nginx`, usamos el servidor Apache (PHP) para servir los archivos estáticos. Para ello, copiamos el contenido de la carpeta `dist` del frontend a la carpeta donde Apache busca los archivos web (/var/www/html).

### Objetivo:
El objetivo es que los archivos estáticos del frontend generados durante la fase de compilación (en la carpeta dist) sean servidos por el servidor Apache del contenedor PHP.

### Cambios clave y por qué se realizaron:

Ahora que los archivos generados del frontend se están sirviendo a través del servidor PHP (Apache), en lugar de usar el entorno de desarrollo de Vite (puerto 3001), accederás a la web desde el puerto 80, que es el puerto estándar para servidores web.

1. Servidor web en producción:

    - Antes, el frontend estaba siendo servido a través de Vite en modo desarrollo. Vite es excelente para desarrollo local, ya que tiene recarga en caliente y muchas otras herramientas útiles para desarrollo rápido.
    - Sin embargo, en un entorno de producción, ya no es necesario usar Vite. En lugar de eso, se genera una versión optimizada y estática del frontend (archivos HTML, CSS, JS), y se sirven estos archivos a través de un servidor web (en este caso, Apache con PHP).
2. Acceso desde el puerto 8080:

    - Al mover los archivos estáticos al servidor Apache, ahora el servidor Apache es el responsable de servir la web. Como Apache está configurado para usar el puerto 8080, accedemos a la aplicación web desde http://localhost:8080.
    - Ya no necesitamos acceder a través del puerto 3001 porque ese puerto estaba siendo usado por Vite para servir la web en modo desarrollo.
3. Producción vs Desarrollo:

    - En desarrollo, Vite sirve la aplicación dinámicamente, y se puede acceder a través del puerto 3001.
    - En producción, se genera una versión estática de la aplicación que se sirve a través del servidor Apache en el puerto 80. Esto es más eficiente para entornos productivos.

### Resumen de acceso:

- En desarrollo: 
```yaml
http://localhost:3001
```
- En producción (con los cambios realizados): 
```yaml
http://localhost:8080
```